In [ ]:
%pip install torchmetrics lightning ipywidgets

### 4.3 Implementing Deep Learning Projects

This notebook presents strategies and tutorials for efficiently running deep learning experiments.

### Working with data in Pandas

[Pandas](https://pandas.pydata.org) is a widely-used Python library for working with tabular data.

The `ucimlrepo` package that we have been using loads data as Pandas DataFrames.  If you wanted to load data from a CSV file directly, you could do `df = pd.read_csv(filename)` to obtain a DataFrame.

If you are working in VS Code, I recommend you install the "Data Wrangler" extension, which provides a great viewer for Pandas DataFrames.  

Here I will load the [California Housing Dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices#) which I downloaded from Kaggle.

In [ ]:
import pandas as pd
url = 'https://www.dropbox.com/scl/fi/cpce5si28h5oqnzfpdipw/housing.csv?rlkey=3di6mggl0v65gpmuo1g1gsuq0&dl=1'
df = pd.read_csv(url)
df

Pandas makes it easy to do things like selecting rows and columns or applying various operations to data in the table.

For example, here we remove rows with missing values with `.dropna()`, and then separate out the predictor variables from the target variable.

In [ ]:
df = df.dropna()
X = df[['longitude','latitude','housing_median_age','total_rooms','total_bedrooms','population','households','median_income','ocean_proximity']]
y = df['median_house_value']

### Data preprocessing

Now let's look at how to preprocess data using scikit learn.

`StandardScaler` applies standard scaling, meaning that we subtract the mean and divide by the std. dev.

`.fit(X)` will compute the mean and std. dev., and `.transform(X)` apply the normalization.  `fit_transform(X)` does both in the same call.

In [ ]:
from sklearn.preprocessing import StandardScaler

quant_cols = ['longitude','latitude','housing_median_age','total_rooms','total_bedrooms','population','households','median_income']
StandardScaler().fit_transform(X[quant_cols])

If we have categorical variables, like the `ocean_proximity` column in our dataset, we need to convert them to numeric before we can use them.  

`OrdinalEncoder` will map categorical variables to ordinals (0,1,2,3,...). This is typicall not a good idea, because the network will, for example, interpret a 2 as twice a 1, which doesn't make sense for most categorical variables.  

A better option is to map each category to its own Boolean column, with a 1 indicating which class the observation belongs to.  This is done using the `OneHotEncoder`.

When we `fit` the `OneHotEncoder`, it determines all of the unique values in the column.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
encoder.fit(X[['ocean_proximity']])
encoder.categories_

Now we can transform the column into zeros and ones.

Note that `OneHotEncoder` produces a sparse matrix, which we convert to dense using `.todense()`.

In [ ]:
encoder.transform(X[['ocean_proximity']]).todense()

Since we likely want to preprocess different columns in different ways, we can combine different preprocessors into a single `ColumnTransformer` object using the `make_column_transformer` function.

Note that we need a separate `OneHotEncoder` object for each categorical column, since each one will have its own list of categories.

The final argument `remainder='drop'` tells the `ColumnTransformer` to ignore any columns that aren't listed in the previous arguments.  The alternative is `passthrough` which will keep the columns without modification.

In [ ]:
from sklearn.compose import make_column_transformer

transformer = make_column_transformer(
    (StandardScaler(),quant_cols),
    (OneHotEncoder(),['ocean_proximity']),
    remainder='drop'
)

transformer.fit(X)

Now we can transform all of the columns in our `DataFrame` at once.

In [ ]:
X_transformed = transformer.transform(X)
X_transformed.shape

In [ ]:
X_transformed

### Train/test split

Before we saw how to use the `sklearn.model_selection.train_test_split` to split our data into train/test splits.  I will show you how to do it with PyTorch datasets, which is a more scalable option since Pandas requires the dataset to be loaded into memory.

Note that the `Dataset` object contains both the features and the targets, so we don't need to have separate variables for `X` and `y`.

In [ ]:
import torch
import numpy as np

First we get our data into torch tensors of the appropriate type and shape.

In [ ]:
X_tensor = torch.tensor(X_transformed).float()

In [ ]:
log_y_tensor = torch.tensor(np.log(y.values)).float().unsqueeze(-1)

Now we make a `TensorDataset` and call `random_split` to split it.

In [ ]:
from torch.utils.data import TensorDataset,random_split
ds = TensorDataset(X_tensor,log_y_tensor)
train_ds,test_ds = random_split(ds,[0.8,0.2])

As usual we will put our datasets into `DataLoader`s for shuffling and batching.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

Now let's set up a simple neural network and get ready to optimize it.

In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Linear(X_transformed.shape[-1],1),
)

loss_fn = nn.MSELoss()

opt = torch.optim.Adam(model.parameters(), lr=3e-4)

### Calculating metrics with Torch Metrics

The Torch Metrics library makes it easy to compute metrics like accuracy over batches.

The concept of the library is that you instantiate a metric object like `Accuracy` or `MeanSquaredError`, call it on each batch, and then at the end of the loop call `compute()` to compute the aggregated metric.

In [ ]:
from torchmetrics.regression import MeanSquaredError

metric = MeanSquaredError()

def compute_metric(model,loader):
    metric.reset()
    for (X_batch,y_batch) in train_loader:
        preds = model(X_batch)
        metric(preds,y_batch)
    return metric.compute()

In [ ]:
from tqdm import tqdm

for epoch in range(10):
    model.train() # put the model into "train" mode
    for (X_batch,y_batch) in tqdm(train_loader,desc=f'epoch {epoch}'):
        opt.zero_grad()

        z = model(X_batch)
        loss = loss_fn(z,y_batch)

        loss.backward()
        opt.step()

    model.eval() # put the model into "eval" mode
    train_mse = compute_metric(model,train_loader)
    test_mse = compute_metric(model,test_loader)

    print(f'train mse: {train_mse}, test mse: {test_mse}')

### Simplifying the process with PyTorch Lightning

Training a network with PyTorch involves a lot of boilerplate code that is repeated across projects.  PyTorch Lightning provides a framework where the boilerplate is taken care of for you, allowing you to focus on writing only the customized parts of your setup.

Here we create a `LightningModule` that will contain the NN model as well as the loss function and optimizer.  The module specifies how to run the model and compute the loss.

In [ ]:
import lightning as L

class LitModel(L.LightningModule):
    def __init__(self):
        super().__init__()

        # define the model
        self.model = nn.Sequential(
            nn.Linear(X_transformed.shape[-1],1),
        )

        # define the loss function
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        preds = self.model(x)
        return preds
    
    def common_step(self, batch, phase):
        X_batch, y_batch = batch
        preds = self.model(X_batch)
        loss = self.loss_fn(preds, y_batch)
        self.log(f"{phase}_loss", loss, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.common_step(batch,'train')

    def validation_step(self, batch, batch_idx):
        return self.common_step(batch,'val')

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=3e-4)
        return optimizer


Now we can create a `Logger` object, to write metrics to a CSV file, and a `Trainer` object which will run the training process.

Note that by specifying the argument `accelerator=gpu` to `Trainer` we can run training on the GPU (if we have one).

In [ ]:
logger = L.pytorch.loggers.CSVLogger(save_dir="./",version='experiment')
trainer = L.Trainer(max_epochs=10,check_val_every_n_epoch=1,logger=logger)

Finally, we create the model and call `fit()`!

In [ ]:
litmodel = LitModel()
trainer.fit(litmodel, train_loader, test_loader)

Here is a little code to read in the metrics and make a plot of train and validation loss.

For more sophisticated logging, you can explore other loggers such as `TensorboardLogger` and `WandbLogger`.

In [ ]:
# read the metrics.csv file
metrics = pd.read_csv('lightning_logs/experiment/metrics.csv')

# grab the train and val loss values
train = metrics['train_loss_epoch'].dropna().values
val = metrics['val_loss'].dropna().values

# make a DataFrame with the values
my_metrics = pd.DataFrame({'epochs':np.arange(len(train)),'train':train,'val':val})

# make a line plot
my_metrics[['train','val']].plot.line(xlabel='epoch',ylabel='loss')